# Skills Assessment: IMDB Sentiment Analysis

**Dataset:** IMDB Movie Reviews (Maas et al., 2011) — 50,000 reviews  
**Task:** Binary sentiment classification — Positive (1) vs Negative (0)  

---

### Overview

This notebook trains a sentiment classifier on 25,000 labeled IMDB movie reviews and evaluates
it on a held-out test set of 25,000 reviews. The pipeline covers HTML-aware text preprocessing,
TF-IDF feature extraction with bigrams, model comparison (Logistic Regression vs Naive Bayes),
hyperparameter tuning via GridSearchCV, evaluation, model persistence, and API submission.

### Learning Objectives
- Preprocess longer, noisier text (HTML tags, punctuation, stop words)
- Apply TF-IDF with bigrams to capture sentiment-carrying phrases
- Compare Logistic Regression and Naive Bayes on a balanced dataset
- Tune hyperparameters with GridSearchCV inside a Pipeline
- Evaluate on a perfectly balanced binary classification task
- Save and submit the best model

---

## Background: Sentiment Analysis

Sentiment analysis maps text to a polarity label — positive or negative. It is a foundational
NLP task with applications in content moderation, brand monitoring, and threat intelligence
(e.g. detecting hostile intent in communications).

### Why Logistic Regression for Text?

Logistic Regression is a strong baseline for text classification:
- Works well with high-dimensional sparse features (TF-IDF produces thousands of features)
- Outputs calibrated probabilities via the sigmoid function
- The `C` parameter controls regularisation — low C = strong regularisation, high C = less
- Typically outperforms Naive Bayes on longer, balanced text like movie reviews

### Why TF-IDF with Bigrams?

Sentiment is often carried by phrases, not individual words:

| Unigram | Bigram |
|---|---|
| "not" (ambiguous) | "not good" → negative |
| "great" (positive) | "not great" → negative |
| "bad" (negative) | "not bad" → positive |

Bigrams capture negation and emphasis that unigrams miss entirely.

### Balanced vs Imbalanced Classification

Unlike the spam lab (87/13 split), IMDB is perfectly balanced — 12,500 positive and 12,500
negative reviews in both train and test sets. This means accuracy is a valid metric here,
though F1-score remains the preferred measure for consistency.

---

## Step 1: Import Required Packages

In [ ]:
import re
import json
import joblib
import requests
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

import nltk
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.pipeline import Pipeline
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import (
    accuracy_score, f1_score, confusion_matrix, classification_report
)

nltk.download('stopwords')

SEED = 42
print("All packages imported successfully.")

## Step 2: Load the Dataset

In [ ]:
with open('train.json', 'r') as f:
    train_data = json.load(f)

with open('test.json', 'r') as f:
    test_data = json.load(f)

train_df = pd.DataFrame(train_data)
test_df  = pd.DataFrame(test_data)

print(f"Train shape : {train_df.shape}")
print(f"Test shape  : {test_df.shape}")
print(f"\nTrain label distribution:")
print(train_df['label'].value_counts().rename({1: 'Positive', 0: 'Negative'}))
print(f"\nSample review:")
print(train_df['text'].iloc[0][:300], '...')

## Step 3: Preprocess the Text

IMDB reviews contain HTML artifacts (`<br />`) from web scraping — these must be stripped
before any other processing. Pipeline:

1. **Strip HTML tags** — remove `<br />` and any other tags
2. **Lowercase**
3. **Remove non-alpha characters** — strip punctuation and numbers
4. **Tokenise**
5. **Remove stop words**
6. **Stem** — reduce words to root form
7. **Rejoin**

In [ ]:
stop_words = set(stopwords.words('english'))
stemmer    = PorterStemmer()

def preprocess_text(text):
    """Full preprocessing: strip HTML → lowercase → clean → tokenise → stop words → stem."""
    # 1. Strip HTML tags
    text = re.sub(r'<[^>]+>', ' ', text)
    # 2. Lowercase
    text = text.lower()
    # 3. Keep only letters and whitespace
    text = re.sub(r'[^a-z\s]', '', text)
    # 4. Tokenise
    tokens = text.split()
    # 5. Remove stop words
    tokens = [w for w in tokens if w not in stop_words]
    # 6. Stem
    tokens = [stemmer.stem(w) for w in tokens]
    # 7. Rejoin
    return ' '.join(tokens)

print("Preprocessing training set...")
train_df['cleaned'] = train_df['text'].apply(preprocess_text)
print("Preprocessing test set...")
test_df['cleaned']  = test_df['text'].apply(preprocess_text)

print("\n=== BEFORE vs AFTER ===")
print(f"BEFORE: {train_df['text'].iloc[0][:200]}")
print(f"\nAFTER:  {train_df['cleaned'].iloc[0][:200]}")

## Step 4: Prepare Features and Labels

The dataset already provides a separate test set — no manual split needed.
We use the full 25k training set to fit the model.

In [ ]:
X_train = train_df['cleaned']
y_train = train_df['label']
X_test  = test_df['cleaned']
y_test  = test_df['label']

print(f"Training samples : {len(X_train)}")
print(f"Test samples     : {len(X_test)}")
print(f"Positive rate — train: {y_train.mean():.1%} | test: {y_test.mean():.1%}")

## Step 5: Build Pipelines and Tune with GridSearchCV

We compare two models:
- **Logistic Regression** — typically stronger on longer, balanced text
- **Multinomial Naive Bayes** — fast baseline, strong on shorter text

Both use TF-IDF with bigrams inside a Pipeline. GridSearchCV with 5-fold CV
optimises on F1-score.

> Note: Logistic Regression grid search may take 2-3 minutes.

In [ ]:
# --- Logistic Regression Pipeline ---
lr_pipeline = Pipeline([
    ('tfidf', TfidfVectorizer(ngram_range=(1, 2), max_df=0.9, min_df=2)),
    ('clf',   LogisticRegression(max_iter=1000, random_state=SEED))
])

lr_params = {
    'tfidf__max_features': [50000, 100000],
    'clf__C':              [0.1, 1.0, 10.0]
}

lr_grid = GridSearchCV(lr_pipeline, lr_params, cv=5, scoring='f1', n_jobs=-1, verbose=1)
lr_grid.fit(X_train, y_train)

print(f"\nLR best params : {lr_grid.best_params_}")
print(f"LR best CV F1  : {lr_grid.best_score_:.4f}")

In [ ]:
# --- Naive Bayes Pipeline ---
nb_pipeline = Pipeline([
    ('tfidf', TfidfVectorizer(ngram_range=(1, 2), max_df=0.9, min_df=2)),
    ('clf',   MultinomialNB())
])

nb_params = {
    'tfidf__max_features': [50000, 100000],
    'clf__alpha':          [0.01, 0.1, 0.5, 1.0]
}

nb_grid = GridSearchCV(nb_pipeline, nb_params, cv=5, scoring='f1', n_jobs=-1, verbose=1)
nb_grid.fit(X_train, y_train)

print(f"\nNB best params : {nb_grid.best_params_}")
print(f"NB best CV F1  : {nb_grid.best_score_:.4f}")

## Step 6: Evaluate and Compare Both Models

In [ ]:
def evaluate_model(name, model, X_test, y_test):
    y_pred = model.predict(X_test)
    print(f"\n{'='*55}")
    print(f" {name}")
    print(f"{'='*55}")
    print(f"Accuracy : {accuracy_score(y_test, y_pred):.4f}")
    print(f"F1-Score : {f1_score(y_test, y_pred):.4f}")
    print("\nClassification Report:")
    print(classification_report(y_test, y_pred, target_names=['Negative', 'Positive']))

    cm = confusion_matrix(y_test, y_pred)
    plt.figure(figsize=(6, 4))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=['Negative', 'Positive'],
                yticklabels=['Negative', 'Positive'])
    plt.title(f'Confusion Matrix — {name}')
    plt.xlabel('Predicted')
    plt.ylabel('Actual')
    plt.tight_layout()
    plt.show()

    return f1_score(y_test, y_pred)

lr_f1 = evaluate_model('Logistic Regression', lr_grid.best_estimator_, X_test, y_test)
nb_f1 = evaluate_model('Naive Bayes',         nb_grid.best_estimator_, X_test, y_test)

In [ ]:
winner     = 'Logistic Regression' if lr_f1 >= nb_f1 else 'Naive Bayes'
best_model = lr_grid.best_estimator_ if winner == 'Logistic Regression' else nb_grid.best_estimator_

print("\n=== Head-to-Head: F1-Score ===")
print(f"Logistic Regression : {lr_f1:.4f}")
print(f"Naive Bayes         : {nb_f1:.4f}")
print(f"Winner              : {winner}")

## Step 7: Most Influential Words

For Logistic Regression, the model coefficients directly show which words push predictions
toward positive or negative sentiment — a useful interpretability check.

In [ ]:
# Extract from LR model
lr_model  = lr_grid.best_estimator_
tfidf_vec = lr_model.named_steps['tfidf']
lr_clf    = lr_model.named_steps['clf']

feature_names = tfidf_vec.get_feature_names_out()
coefs         = lr_clf.coef_[0]

top_n    = 15
top_pos  = np.argsort(coefs)[-top_n:][::-1]
top_neg  = np.argsort(coefs)[:top_n]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].barh([feature_names[i] for i in top_pos[::-1]],
             [coefs[i] for i in top_pos[::-1]], color='steelblue')
axes[0].set_title('Top Positive Sentiment Words')
axes[0].set_xlabel('Coefficient')

axes[1].barh([feature_names[i] for i in top_neg],
             [coefs[i] for i in top_neg], color='salmon')
axes[1].set_title('Top Negative Sentiment Words')
axes[1].set_xlabel('Coefficient')

plt.tight_layout()
plt.show()

## Step 8: Save and Reload the Best Model

In [ ]:
model_path = 'sentiment_model.joblib'
joblib.dump(best_model, model_path)
print(f"Model saved → {model_path}")

loaded_model = joblib.load(model_path)
print("Model reloaded successfully.")

assert (loaded_model.predict(X_test) == best_model.predict(X_test)).all()
print("Sanity check passed.")

## Step 9: Live Classification

In [ ]:
def classify_review(text, model=loaded_model):
    """Classify a raw movie review as Positive or Negative."""
    cleaned = preprocess_text(text)
    pred    = model.predict([cleaned])[0]
    prob    = model.predict_proba([cleaned])[0]
    label   = 'Positive' if pred == 1 else 'Negative'
    return f"{label} (confidence: {max(prob):.2%})"

test_reviews = [
    "This film was absolutely brilliant. The acting was superb and the story kept me hooked throughout.",
    "Complete waste of time. Terrible acting, no plot, and the ending made no sense whatsoever.",
    "Not bad, but not great either. Some good moments but overall pretty forgettable.",
    "One of the best movies I have seen in years. A masterpiece of storytelling.",
    "I fell asleep halfway through. Boring, predictable, and painfully slow."
]

print("=== Live Sentiment Classification ===")
for review in test_reviews:
    print(f"\nReview    : {review[:80]}...")
    print(f"Sentiment : {classify_review(review)}")

## Step 10: Submit to HTB Evaluation API

In [ ]:
url             = "http://localhost:8000/api/upload"
model_file_path = "sentiment_model.joblib"

try:
    with open(model_file_path, "rb") as model_file:
        files    = {"model": model_file}
        response = requests.post(url, files=files)
    print(f"Status : {response.status_code}")
    print(json.dumps(response.json(), indent=4))
except Exception as e:
    print(f"Error: {e}")

---

## Personal Analysis

### Why Logistic Regression Outperforms Naive Bayes on IMDB

Naive Bayes assumes word independence — a reasonable approximation for short spam messages
but increasingly wrong for longer reviews where context and word co-occurrence carry meaning.
Logistic Regression makes no independence assumption; it learns weights for each feature
directly from the data, allowing it to capture more nuanced signals in longer text.
On IMDB, Logistic Regression with TF-IDF typically achieves ~89-91% accuracy vs ~85-87% for Naive Bayes.

### The Role of Bigrams in Sentiment

Negation is the key reason bigrams matter for sentiment analysis. "not good", "not bad",
"not great" all carry opposite polarity to their unigram components. Without bigrams,
the model sees "not" and "good" separately — both of which appear frequently in both
positive and negative reviews — and loses the negation signal entirely.

### Balanced Dataset vs Spam Lab

IMDB is perfectly balanced at 50/50, which means accuracy is a valid primary metric —
unlike the spam lab where the 87/13 split made accuracy misleading. A classifier that
labels everything as positive achieves only 50% accuracy here, not 87%. This is why
dataset balance matters so much when choosing evaluation metrics.

### HTML Preprocessing

The `<br />` tags in the raw reviews come from HTML-formatted content scraped from IMDB.
Without stripping them, the vectoriser would treat `br` as a high-frequency token appearing
in nearly every review — useless for classification and consuming vocabulary budget.
A broader HTML strip regex (`<[^>]+>`) handles any tag, not just line breaks.

### Logistic Regression Coefficients as Interpretability

Unlike tree-based models or neural networks, Logistic Regression coefficients are directly
interpretable — positive coefficients push toward positive sentiment, negative toward negative.
This matters in security contexts where analysts need to understand *why* a model made a
decision, not just what it decided. The top positive words (excellent, brilliant, masterpiece)
and top negative words (waste, terrible, awful) confirm the model is learning real sentiment
signals rather than spurious correlations.